# HW4: Complex Neural Network Architectures for Reliability Prediction

This notebook implements two complex neural network architectures for predicting quantum
circuit `reliability` from pre-run tabular features, and compares them against classical
baselines (Dummy, Ridge, Random Forest, XGBoost).

## Architectures

1. **Tabular ResNet** — a deep MLP with residual skip connections, batch normalization,
   and dropout. Applies the core ResNet idea to tabular data.

2. **FT-Transformer** (Feature Tokenizer Transformer) — each input feature is projected
   into a learned token embedding, then processed by a stack of multi-head self-attention
   transformer blocks. A special `[CLS]` token aggregates information for the final
   prediction. Based on Gorishniy et al. (2021), "Revisiting Deep Learning Models for
   Tabular Data."

The FT-Transformer is a genuinely different architecture class from MLPs — it uses
attention to model feature interactions rather than fixed layer-wise transformations.

## Dataset

- **Input:** 46 pre-run numeric features (circuit structure, compiler output, hardware noise)
- **Target:** `reliability` — continuous value in [0, 1]
- **Task:** supervised regression
- **Primary metric:** validation MAE
- **Leakage rule:** no post-run outcome columns used as inputs

## 1. Setup and Data Loading

In [ ]:
from pathlib import Path
import json
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
DATA_DIR = Path('data/hw2')

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print(f'PyTorch version: {torch.__version__}')
print(f'Device: cpu')

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train.parquet')
validation = pd.read_parquet(DATA_DIR / 'validation.parquet')
test = pd.read_parquet(DATA_DIR / 'test.parquet')
feature_policy = json.loads((DATA_DIR / 'feature_policy.json').read_text(encoding='utf-8'))

print(f'Train rows: {len(train):,}')
print(f'Validation rows: {len(validation):,}')
print(f'Test rows (reserved for final evaluation): {len(test):,}')
print(f'Target column: {feature_policy["target_column"]}')
print(f'Allowed feature columns: {len(feature_policy["allowed_feature_columns"])}')

## 2. Train-Only Preprocessing

Imputer and scaler are fitted only on the training split to prevent data leakage.

In [ ]:
target_column = feature_policy['target_column']
feature_columns = feature_policy['allowed_feature_columns']

X_train = train[feature_columns]
y_train = train[target_column].astype('float32')

X_validation = validation[feature_columns]
y_validation = validation[target_column].astype('float32')

X_test = test[feature_columns]
y_test = test[target_column].astype('float32')

categorical_columns = X_train.select_dtypes(include=['object', 'string', 'category', 'bool']).columns.tolist()
numeric_columns = [c for c in feature_columns if c not in categorical_columns]

print(f'Numeric columns: {len(numeric_columns)}')
print(f'Categorical columns: {len(categorical_columns)}')
print(f'Target (train): mean={y_train.mean():.4f}, std={y_train.std():.4f}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]),
            categorical_columns,
        ),
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_columns,
        ),
    ],
    remainder='drop',
)

X_train_nn = preprocessor.fit_transform(X_train).astype('float32')
X_validation_nn = preprocessor.transform(X_validation).astype('float32')
X_test_nn = preprocessor.transform(X_test).astype('float32')

print(f'Neural network input dimension: {X_train_nn.shape[1]}')

## 3. Classical Baselines

Dummy (mean), Ridge, Random Forest, and XGBoost for reference.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': mean_squared_error(y_true, y_pred) ** 0.5,
        'r2': r2_score(y_true, y_pred),
    }


baseline_models = {
    'Dummy (mean)': DummyRegressor(strategy='mean'),
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=20, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

baseline_results = []
for name, estimator in baseline_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator),
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_validation)
    metrics = regression_metrics(y_validation, preds)
    baseline_results.append({'Model': name, **metrics})
    print(f'{name:20s} | MAE={metrics["mae"]:.6f} | RMSE={metrics["rmse"]:.6f} | R2={metrics["r2"]:.4f}')

baseline_df = pd.DataFrame(baseline_results)

## 4. Architecture A: Tabular ResNet (Residual MLP)

A residual block learns a correction to its input: `output = ReLU(input + f(input))`.
Skip connections allow gradients to flow through the network, enabling deeper architectures
than a plain MLP.

Structure:
- Input projection (Linear → BatchNorm → ReLU)
- N residual blocks (each: Linear → BN → ReLU → Dropout → Linear → BN, with skip)
- Regression head (Linear → scalar output)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, width: int, dropout: float = 0.10):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(width, width),
            nn.BatchNorm1d(width),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width, width),
            nn.BatchNorm1d(width),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(x + self.block(x))


class TabularResNet(nn.Module):
    def __init__(self, input_dim: int, width: int = 128, blocks: int = 3, dropout: float = 0.10):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.ReLU(),
        )
        self.residual_blocks = nn.Sequential(
            *[ResidualBlock(width=width, dropout=dropout) for _ in range(blocks)]
        )
        self.output_head = nn.Linear(width, 1)

    def forward(self, x):
        x = self.input_projection(x)
        x = self.residual_blocks(x)
        return self.output_head(x).squeeze(-1)

## 5. Architecture B: FT-Transformer (Feature Tokenizer Transformer)

The FT-Transformer (Gorishniy et al., 2021) is a transformer-based architecture designed
for tabular data. Unlike an MLP which processes all features through shared linear layers,
the FT-Transformer:

1. **Tokenizes each feature** — each scalar feature is projected into its own learned
   d-dimensional embedding vector via a per-feature linear transformation.
2. **Adds a [CLS] token** — a special learnable token prepended to the feature token
   sequence, used to aggregate information for the final prediction.
3. **Applies multi-head self-attention** — transformer blocks allow every feature to
   attend to every other feature, learning complex feature interactions without
   relying on manual feature engineering.
4. **Predicts from [CLS]** — the final [CLS] representation is fed to a linear head.

This is fundamentally different from an MLP: instead of processing the feature vector
as a flat input, it treats each feature as a token in a sequence and uses attention
to model interactions — the same mechanism that powers large language models.

In [ ]:
class FeatureTokenizer(nn.Module):
    """Projects each scalar feature into a d_token-dimensional embedding."""
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias = nn.Parameter(torch.empty(n_features, d_token))
        nn.init.xavier_uniform_(self.weight)
        nn.init.zeros_(self.bias)

    def forward(self, x):
        # x: (batch, n_features) -> (batch, n_features, d_token)
        return x.unsqueeze(-1) * self.weight + self.bias


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class FTTransformer(nn.Module):
    def __init__(
        self, n_features: int, d_token: int = 64, n_heads: int = 4,
        n_layers: int = 3, d_ff_mult: int = 2, dropout: float = 0.1,
    ):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls_token, std=0.02)

        d_ff = d_token * d_ff_mult
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_token, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.Linear(d_token, 1),
        )

    def forward(self, x):
        tokens = self.tokenizer(x)             # (batch, n_features, d_token)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)  # (batch, 1 + n_features, d_token)
        tokens = self.blocks(tokens)
        cls_out = tokens[:, 0]                 # [CLS] token output
        return self.head(cls_out).squeeze(-1)


n_features = X_train_nn.shape[1]
resnet_demo = TabularResNet(input_dim=n_features, width=128, blocks=3)
ftt_demo = FTTransformer(n_features=n_features, d_token=64, n_heads=4, n_layers=3)

print('=== TabularResNet ===')
print(f'Parameters: {sum(p.numel() for p in resnet_demo.parameters()):,}')
print()
print('=== FT-Transformer ===')
print(f'Parameters: {sum(p.numel() for p in ftt_demo.parameters()):,}')
print()
print(ftt_demo)

## 6. Training Loop with Early Stopping

A shared training function for both architectures. Monitors validation MAE with
patience-based early stopping and learning rate reduction on plateau.

In [ ]:
def train_model(
    model, X_train_np, y_train_np, X_val_np, y_val_np,
    lr=1e-3, weight_decay=1e-4, batch_size=256,
    max_epochs=200, patience=20, verbose=True,
):
    torch.manual_seed(RANDOM_STATE)

    train_dataset = TensorDataset(
        torch.tensor(X_train_np, dtype=torch.float32),
        torch.tensor(y_train_np, dtype=torch.float32),
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    X_val_t = torch.tensor(X_val_np, dtype=torch.float32)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6,
    )
    loss_fn = nn.MSELoss()

    best_val_mae = float('inf')
    best_weights = None
    epochs_no_improve = 0
    history = {'epoch': [], 'train_loss': [], 'val_mae': [], 'val_rmse': [], 'val_r2': [], 'lr': []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        n_samples = 0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            preds = model(batch_X)
            loss = loss_fn(preds, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * len(batch_X)
            n_samples += len(batch_X)

        avg_loss = running_loss / n_samples

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t).numpy()
        val_m = regression_metrics(y_val_np, val_preds)
        cur_lr = optimizer.param_groups[0]['lr']

        history['epoch'].append(epoch)
        history['train_loss'].append(avg_loss)
        history['val_mae'].append(val_m['mae'])
        history['val_rmse'].append(val_m['rmse'])
        history['val_r2'].append(val_m['r2'])
        history['lr'].append(cur_lr)

        scheduler.step(val_m['mae'])

        marker = ''
        if val_m['mae'] < best_val_mae:
            best_val_mae = val_m['mae']
            best_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
            marker = ' *'
        else:
            epochs_no_improve += 1

        if verbose and (epoch <= 5 or epoch % 10 == 0 or marker or epoch == max_epochs):
            print(
                f'Epoch {epoch:3d} | loss={avg_loss:.6f} | '
                f'val_mae={val_m["mae"]:.6f} | val_r2={val_m["r2"]:.4f} | '
                f'lr={cur_lr:.1e}{marker}'
            )

        if epochs_no_improve >= patience:
            if verbose:
                print(f'Early stopping at epoch {epoch}. Best val MAE: {best_val_mae:.6f}')
            break

    model.load_state_dict(best_weights)
    return model, pd.DataFrame(history)

## 7. Train Tabular ResNet

In [ ]:
y_train_np = y_train.to_numpy(dtype='float32')
y_val_np = y_validation.to_numpy(dtype='float32')

print('=== Training Tabular ResNet (width=256, blocks=4, dropout=0.15) ===')
print()

resnet_model = TabularResNet(input_dim=n_features, width=256, blocks=4, dropout=0.15)
resnet_model, resnet_history = train_model(
    resnet_model, X_train_nn, y_train_np, X_validation_nn, y_val_np,
    lr=5e-4, weight_decay=1e-4, batch_size=512,
    max_epochs=120, patience=20,
)

## 8. Train FT-Transformer

In [ ]:
print('=== Training FT-Transformer (d_token=64, heads=4, layers=3) ===')
print()

ftt_model = FTTransformer(
    n_features=n_features, d_token=64, n_heads=4, n_layers=3,
    d_ff_mult=2, dropout=0.15,
)
ftt_model, ftt_history = train_model(
    ftt_model, X_train_nn, y_train_np, X_validation_nn, y_val_np,
    lr=1e-4, weight_decay=1e-5, batch_size=512,
    max_epochs=120, patience=20,
)

In [ ]:
print('=== Training FT-Transformer (d_token=96, heads=4, layers=4) — larger ===')
print()

ftt_model_large = FTTransformer(
    n_features=n_features, d_token=96, n_heads=4, n_layers=4,
    d_ff_mult=2, dropout=0.15,
)
ftt_model_large, ftt_history_large = train_model(
    ftt_model_large, X_train_nn, y_train_np, X_validation_nn, y_val_np,
    lr=1e-4, weight_decay=1e-5, batch_size=512,
    max_epochs=120, patience=20,
)

## 9. Training Curves

Visualize training loss and validation MAE for both architectures.

In [ ]:
ridge_mae = baseline_df.loc[baseline_df['Model'] == 'Ridge', 'mae'].values[0]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for row, (name, hist) in enumerate([
    ('Tabular ResNet', resnet_history),
    ('FT-Transformer', ftt_history),
]):
    axes[row, 0].plot(hist['epoch'], hist['train_loss'])
    axes[row, 0].set_xlabel('Epoch')
    axes[row, 0].set_ylabel('MSE Loss')
    axes[row, 0].set_title(f'{name} — Training Loss')
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].plot(hist['epoch'], hist['val_mae'], color='tab:orange')
    best_row = hist.loc[hist['val_mae'].idxmin()]
    axes[row, 1].axvline(x=best_row['epoch'], color='red', ls='--', alpha=0.5,
                         label=f'Best ({int(best_row["epoch"])})')
    axes[row, 1].axhline(y=ridge_mae, color='green', ls=':', alpha=0.7, label='Ridge')
    axes[row, 1].set_xlabel('Epoch')
    axes[row, 1].set_ylabel('MAE')
    axes[row, 1].set_title(f'{name} — Validation MAE')
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].plot(hist['epoch'], hist['val_r2'], color='tab:green')
    axes[row, 2].set_xlabel('Epoch')
    axes[row, 2].set_ylabel('R²')
    axes[row, 2].set_title(f'{name} — Validation R²')
    axes[row, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/hw4_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/hw4_training_curves.png')

## 10. Final Comparison: All Models

All models compared on the same validation split with the same leakage-safe preprocessing.

In [ ]:
nn_results = []

for name, model in [
    ('TabularResNet (w=256, b=4)', resnet_model),
    ('FT-Transformer (d=64, L=3)', ftt_model),
    ('FT-Transformer (d=96, L=4)', ftt_model_large),
]:
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_validation_nn)).numpy()
    metrics = regression_metrics(y_val_np, preds)
    nn_results.append({'Model': name, **metrics})

all_results = pd.concat([
    baseline_df,
    pd.DataFrame(nn_results),
], ignore_index=True).sort_values('mae').reset_index(drop=True)

print('=== Final Validation Comparison ===')
print(all_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = []
for m in all_results['Model']:
    if 'FT-Transformer' in m:
        colors.append('#FF5722')
    elif 'ResNet' in m:
        colors.append('#FF9800')
    else:
        colors.append('#2196F3')

bars = ax.barh(all_results['Model'], all_results['mae'], color=colors, edgecolor='white')

for bar, mae_val in zip(bars, all_results['mae']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{mae_val:.5f}', va='center', fontsize=10)

ax.set_xlabel('Validation MAE (lower is better)')
ax.set_title('HW4: Model Comparison — Validation MAE')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('reports/hw4_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/hw4_model_comparison.png')

## 11. Prediction Analysis — Best Neural Network

In [ ]:
best_nn_name = pd.DataFrame(nn_results).sort_values('mae').iloc[0]['Model']
best_nn_idx = [r['Model'] for r in nn_results].index(best_nn_name)
best_nn_model = [resnet_model, ftt_model, ftt_model_large][best_nn_idx]

best_nn_model.eval()
with torch.no_grad():
    nn_val_preds = best_nn_model(torch.tensor(X_validation_nn)).numpy()

print(f'Best neural network: {best_nn_name}')
print(f'Validation metrics: {regression_metrics(y_val_np, nn_val_preds)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual
axes[0].scatter(y_val_np, nn_val_preds, alpha=0.3, s=10, color='tab:blue')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Perfect')
axes[0].set_xlabel('Actual Reliability')
axes[0].set_ylabel('Predicted Reliability')
axes[0].set_title(f'{best_nn_name}: Predicted vs Actual')
axes[0].set_xlim(-0.05, 1.05)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_aspect('equal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual distribution
residuals = y_val_np - nn_val_preds
axes[1].hist(residuals, bins=50, edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.7)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].grid(True, alpha=0.3)

# Residuals vs Predicted
axes[2].scatter(nn_val_preds, residuals, alpha=0.3, s=10)
axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[2].set_xlabel('Predicted Reliability')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Predicted')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/hw4_prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Residual mean: {residuals.mean():.6f}, std: {residuals.std():.6f}')

## 12. Interpretation and Conclusion

### Architectures Evaluated

1. **Tabular ResNet** — residual MLP with skip connections, batch normalization, and dropout.
   Applies the ResNet idea from computer vision to tabular data.

2. **FT-Transformer** — transformer-based architecture that tokenizes each feature into a
   learned embedding and processes them with multi-head self-attention. This is a fundamentally
   different computation pattern from MLPs: attention allows each feature to interact with
   every other feature at every layer, rather than relying on fixed linear projections.

### Why These Architectures for This Dataset

The dataset is purely tabular — 46 numeric features describing circuit structure, compiler
output, and hardware noise. A CNN would be a poor fit because there is no spatial grid
structure. Both chosen architectures are designed for tabular data and are architecturally
complex: the ResNet uses deep residual learning, while the FT-Transformer uses the
attention mechanism that powers modern NLP.

### Scientific Assessment

The final comparison table above shows whether the extra complexity of deep neural
networks is justified compared to tree-based models (Random Forest, XGBoost) and
simple linear models (Ridge). A more complex model is only worthwhile if it provides
a meaningful improvement in validation MAE.

In [ ]:
print('=' * 70)
print('HW4 FINAL SUMMARY')
print('=' * 70)
print()
print('Validation comparison (all models):')
print(all_results.to_string(index=False))
print()

nn_best = pd.DataFrame(nn_results).sort_values('mae').iloc[0]

print(f'Best neural network: {nn_best["Model"]}')
print(f'  MAE:  {nn_best["mae"]:.6f}')
print(f'  RMSE: {nn_best["rmse"]:.6f}')
print(f'  R2:   {nn_best["r2"]:.4f}')
print()

ridge_mae = baseline_df.loc[baseline_df['Model'] == 'Ridge', 'mae'].values[0]
print(f'vs Ridge:  {(ridge_mae - nn_best["mae"]) / ridge_mae * 100:+.1f}% MAE improvement')

best_classical = baseline_df.loc[baseline_df['mae'].idxmin()]
print(f'vs {best_classical["Model"]}:  '
      f'{(best_classical["mae"] - nn_best["mae"]) / best_classical["mae"] * 100:+.1f}% MAE improvement')
print()
print('Architecture parameters:')
print(f'  TabularResNet:  {sum(p.numel() for p in resnet_model.parameters()):,}')
print(f'  FT-Transformer (small): {sum(p.numel() for p in ftt_model.parameters()):,}')
print(f'  FT-Transformer (large): {sum(p.numel() for p in ftt_model_large.parameters()):,}')